# 이전 4시간 구성 · 선택 복습
현재 필수 수업은 [교재 안내](../../README.md)를 따릅니다. 이 파일은 이전 상세 실습 보존본입니다.


# LangChain 프롬프트와 대화 이력

**목표:** LangChain 공식 API의 입력·반환값과 실행 위치를 직접 설명합니다.

[교재 안내](../../README.md) · [API 상세 해설](../../LANGCHAIN_API.md) · [슬라이드 23~36쪽](../../../슬라이드/강의자료_2일차_v2.pdf#page=23)

**커널:** Python (Day2 LangChain). 설치·커널 선택을 완료한 뒤 코드 셀을 순서대로 실행하세요. 본문은 LangChain API를 직접 사용합니다. 모의 예제와 실제 요청을 구분하며 정답 출력은 미리 넣지 않습니다.

60분 동안 API 설명과 실행 확인을 번갈아 진행합니다. 상세 옵션·추가 비교는 복습 자료로도 사용합니다. 마지막 시간에는 TODO를 진행합니다.
## 실습 진행 지도

![02 실습 전체 순서](../../assets/02-flow-overview.svg)

먼저 준비 셀 3개를 실행하고 프롬프트의 입력·반환 타입을 확인합니다. 2·3절은 같은 질문을 이력 없이/있게 보내는 실험입니다. 4절의 단계 지시를 본 뒤 4-1~4-4절에서 박스를 Gemma 3로 판단하고 Gemma 4를 추가합니다. 5절은 선택 실험, 마지막은 직접 코드를 작성하는 TODO입니다.

한 셀을 실행한 뒤 출력이 예상과 맞는지 읽고 다음 셀로 넘어가세요. 모델 응답이 다르게 나와도 정답 문장으로 바꾸지 않고 원문·상태·시간을 남깁니다.


In [ ]:
from pathlib import Path
import json, os, sys, time
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / '04.LLM_v2').is_dir()), None)
if ROOT is None:
    raise RuntimeError('배포 폴더 안에서 노트북을 여세요.')
COURSE = ROOT / '04.LLM_v2'
sys.path.insert(0, str(COURSE))
print('Python:', sys.executable)
print('교재:', COURSE)
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda

In [ ]:
model = ChatOllama(
    model="gemma3:1b",
    base_url="http://127.0.0.1:11434",
    temperature=0,
    seed=2026,
    num_ctx=4096,
    num_predict=256,
    keep_alive="5m",
    client_kwargs={"timeout": 180},
)
print(model.model, model.base_url)

In [ ]:
BASE_RULE = '한국어로 간결하게 답하세요. 주어진 정보가 부족하면 부족하다고 밝히세요.'
STEP_RULE = ('조건을 먼저 확인하고 문제를 단계별로 검토하세요. '
             '출력은 결론과 확인 가능한 짧은 근거 2개 이내로 작성하세요.')

## 1. 두 입력 조건을 독립적으로 비교

기억은 최근 human/ai 메시지를 다시 보내는 것이고, 단계별 지시는 system 내용에 조건 검토 요구를 추가하는 것입니다. LangChain에 memory=True라는 공통 모델 옵션이 있는 것이 아닙니다. 아래 실습의 memory·steps는 우리가 정한 비교 변수입니다.

### LangChain API · ChatPromptTemplate.from_messages

`ChatPromptTemplate.from_messages(...)`는 메시지 구조를 재사용할 프롬프트 객체로 만듭니다. `(역할, 템플릿 문자열)`을 나열하고 문자열 안의 `{question}` 같은 자리를 실행 시 사전 값으로 채웁니다. 이 사전의 키는 템플릿 변수명과 일치해야 합니다.

`prompt.invoke({"question": ...})`는 **ChatPromptValue**를 반환합니다. 여기까지는 모델을 호출하지 않습니다. `value.to_messages()`는 SystemMessage·HumanMessage 등의 실제 메시지 목록을 반환합니다. 요청 전에 입력을 확인할 때 사용하세요.

이미 만들어진 SystemMessage 객체를 from_messages에 넣으면 그 내용은 고정 메시지입니다. 변수 치환이 필요한 문자열은 튜플의 템플릿으로 전달하세요. 기본 템플릿에서 JSON 중괄호를 문자로 쓰려면 이중 중괄호로 표현하거나 JSON 문자열을 별도 변수 값으로 전달합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하게 답하세요."),
    ("human", "{question}"),
])
value = prompt.invoke({"question": "센서란?"})
print(type(value).__name__)
print([(m.type, m.content) for m in value.to_messages()])

### LangChain API · MessagesPlaceholder는 저장소가 아닙니다

`MessagesPlaceholder("history")`는 **메시지 목록을 넣을 자리**입니다. 과거 대화를 저장하거나 서버에서 읽어오는 API가 아닙니다. 프로그램이 history 목록을 관리하고 invoke의 입력 사전에 전달해야 합니다.

기본 optional=False에서는 첫 요청에도 `history: []`가 필요합니다. optional=True를 선택하면 생략할 수 있지만 수업에서는 빈 목록을 명시합니다. 문자열로 붙인 대화가 아니라 역할이 보존된 HumanMessage·AIMessage 목록을 전달합니다.

예제의 예상 메시지 순서는 system → 이전 human → 이전 ai → 이번 human입니다. 최근 max_pairs개 쌍을 유지하려면 Python에서 history[-2 * max_pairs:]를 선택합니다. MessagesPlaceholder의 n_messages는 메시지 개수 기준이므로 대화 쌍 제한과 혼동하지 마세요.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "한국어로 간결하게 답하세요."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
sample_history = [HumanMessage("코드 BLUE-27"), AIMessage("확인")]
value = history_prompt.invoke({
    "history": sample_history, "question": "작업 코드는?",
})
print([m.type for m in value.to_messages()])

### LangChain API · 파이프 연산자와 RunnableSequence

LangChain의 `|`는 왼쪽 단계의 반환값을 오른쪽 단계의 입력으로 연결합니다. `chain = prompt | model`은 RunnableSequence를 구성하며 이 줄만으로 모델을 호출하지 않습니다. `chain.invoke(data)`를 실행하면 data → ChatPromptValue → AIMessage 순서로 진행됩니다.

`model.invoke(...)`에는 메시지를, `chain.invoke(...)`에는 첫 단계인 prompt가 요구하는 변수 사전을 넣습니다. 입력이 무엇인지 외우기보다 **첫 단계의 입력 계약**을 읽으세요. 연결 양쪽 타입이 맞아야 합니다.

아래 fake_model은 구조 확인을 위한 모의 응답입니다. 실제 요청 없이 체인의 입력과 반환을 확인합니다. 실제 실습에서는 같은 자리에 model을 연결합니다. 두 방식으로 모두 invoke하면 모델도 두 번 요청되므로 비교 설명용 코드와 실제 요청 셀을 구분합니다.
![프롬프트와 실제 모델 요청 구분](../../assets/02-flow-messages.svg)

**읽는 순서:** 입력 사전의 키 → 템플릿 변수 → to_messages()의 역할·순서 → AIMessage의 content입니다. 아래 모의 체인은 서버 요청 없이 연결을 확인합니다. 마지막 `chain = history_prompt | model`은 실제 모델 연결을 준비하지만 그 줄에서 요청하지 않습니다.


In [ ]:
from langchain_core.runnables import RunnableLambda
fake_model = RunnableLambda(lambda value: AIMessage(
    content="모의 응답: " + value.to_messages()[-1].content
))
demo_chain = prompt | fake_model
demo_reply = demo_chain.invoke({"question": "센서란?"})
print(type(demo_chain).__name__, demo_reply.content)
chain = history_prompt | model  # 연결만 구성합니다.

## 2. 기억 없는 두 턴

매 요청에 history=[]를 주면 앞 질문·답이 다음 입력에 포함되지 않습니다. `chain.invoke`의 첫 단계는 프롬프트이므로 변수 사전을 전달합니다. 실제 입력 메시지를 먼저 출력하고 응답을 비교하세요.
**실습 절차:** 첫 질문에 작업 코드를 알려주고 응답을 읽습니다. 두 번째 요청의 메시지 목록에 첫 질문이 없는지 확인합니다. 그다음 모델 답을 읽어, 코드가 없다는 응답인지 근거 없는 추측인지 구분하세요. 셀을 재실행하면 두 요청이 다시 나갑니다.


In [ ]:
for question in ['작업 코드가 BLUE-27입니다. 확인했다고 답하세요.', '앞서 알려준 작업 코드는 무엇인가요?']:
    inputs = {'history': [], 'question': question}
    print([(m.type, m.content) for m in history_prompt.invoke(inputs).to_messages()])
    response = chain.invoke(inputs)
    print(response.content)

## 3. 기억을 메시지 목록으로 직접 전달

최근 3쌍은 최대 6개 메시지입니다. 정상 반환을 확인한 뒤 history에 질문과 답을 한 쌍으로 추가합니다. 이번 질문은 프롬프트의 마지막 human 자리에 들어가므로 history에 미리 추가하지 않습니다.
![대화 이력의 선택과 갱신 흐름](../../assets/02-flow-history.svg)

**실행 전:** `history=[]`에서 시작하며 max_pairs=3이면 최대 메시지 6개를 유지합니다. 첫 턴 입력에는 이력이 없고, 두 번째 입력에는 정상 완료한 첫 human/ai 쌍이 있어야 합니다.

**코드 읽기:** for 안에서 recent 선택 → inputs 구성 → prompt 내용 출력 → try에서 요청·반환 검사 → except 또는 else 순서로 따라갑니다. `else`는 try에서 예외가 발생하지 않았을 때만 실행됩니다. 이번 질문을 요청 전에 history에 추가하지 않는 이유는 실패한 질문만 남는 것을 막기 위해서입니다.

**실행 후:** 마지막 역할 목록이 human/ai 쌍인지 확인하세요. 답을 맞혔는지와 이력에 필요한 정보가 들어 있는지를 따로 확인합니다.


In [ ]:
history = []
max_pairs = 3
for question in ['작업 코드가 BLUE-27입니다. 확인했다고 답하세요.', '앞서 알려준 작업 코드는 무엇인가요?']:
    recent = history[-2 * max_pairs:]
    inputs = {'history': recent, 'question': question}
    pending = history_prompt.invoke(inputs)
    print([(m.type, m.content) for m in pending.to_messages()])
    try:
        response = chain.invoke(inputs)
        if not response.content or response.response_metadata.get('done_reason') == 'length':
            raise ValueError('빈 응답 또는 출력 상한으로 잘린 응답입니다.')
    except Exception as exc:
        print('이력 유지:', type(exc).__name__, str(exc))
    else:
        history = (recent + [HumanMessage(question), AIMessage(response.content)])[-2 * max_pairs:]
        print(response.content)
print([m.type for m in history])

### LangChain API · API 오류와 대화 이력 갱신 순서

변수 누락으로 발생하는 템플릿 오류는 모델 호출 전 오류입니다. 연결 실패·타임아웃은 모델 호출 단계에서 발생할 수 있습니다. 빈 응답이나 length 종료는 응답을 받은 후 확인해야 합니다. 답변이 왔다는 사실과 정답이라는 판단도 구분합니다.

안전한 학습 흐름은 **최근 이력 선택 → 메시지 확인 → invoke → 반환 확인 → 이력 추가**입니다. 실패한 질문을 먼저 history에 추가하면 다음 입력에 짝이 맞지 않는 대화가 남을 수 있습니다. 오류를 잡았다는 이유로 성공 답변을 만들지 마세요.

아래에서는 필수 question을 생략해 오류를 관찰합니다. 실제 서버를 호출하지 않습니다. 이어지는 실습은 정상 응답일 때에만 human/ai 한 쌍을 추가하고, 예외일 때 기존 history를 보존합니다.

In [ ]:
try:
    history_prompt.invoke({"history": []})
except KeyError as exc:
    print("모델 호출 전 변수 누락:", type(exc).__name__)
print("필요한 변수:", history_prompt.input_variables)

## 4. 단계별 지시를 system에 추가

steps는 API의 추론 엔진 스위치가 아닙니다. 아래는 system 문자열만 바꾼 프롬프트입니다. 조건 확인 후 결론과 짧은 근거를 요청합니다.
**실습 절차:** system 문자열에서 BASE_RULE과 STEP_RULE이 합쳐졌는지 읽고, 사용자 질문에 확인 완료 조건과 현재 가림 상태가 모두 있는지 확인합니다. 실행 후 결론과 짧은 근거가 조건에 맞는지 표시하세요. `guided_chain`은 앞의 `chain`과 다른 객체이며 이력 자리도 없는 독립 요청입니다.


In [ ]:
guided_prompt = ChatPromptTemplate.from_messages([
    ('system', BASE_RULE + ' ' + STEP_RULE),
    ('human', '{question}'),
])
guided_chain = guided_prompt | model
question = ('확인 완료 조건은 빨간 블록 1개와 파란 블록 1개가 보이는 것입니다. '
            '파란 블록은 상자에 가려져 있습니다. 확인 완료로 기록할까요?')
response = guided_chain.invoke({'question': question})
print(response.content)

### 4-1. 빨간 박스와 파란 박스의 판단 문제 준비

정상·가림·누락·관찰 정보 없음의 **박스 문제 네 개**만 사용합니다. 먼저 `gemma3:1b`로 판단하고 답과 근거를 읽은 뒤, 같은 문제를 `gemma4:e2b-it-qat`에 보내 비교합니다.

모든 문제는 문장으로 주어진 관찰을 판단하는 독립 요청입니다. 이력은 빈 목록으로 시작하며 이미지 입력은 없습니다. 판정 규칙은 두 모델에 동일하게 전달하고, 사례별 정답은 평가에만 사용합니다.
![박스 관찰과 두 모델의 실행 순서](../../assets/02-flow-boxes.svg)

**실행 전 예상:** 네 관찰을 먼저 읽고 각 expected가 왜 그 판정인지 설명합니다. `bench_cases`는 문제 사전 네 개를 담은 목록입니다. id는 기록을 찾는 이름, question은 실제 입력, history는 빈 목록, expected는 사람 평가 기준입니다.

**입력 확인:** 아래 준비 셀은 모델을 호출하지 않습니다. 첫 문제의 system/human 메시지가 출력되는지 확인하세요. expected는 inputs에 넣지 않으므로 모델에 정답을 미리 알려주지 않습니다.


In [ ]:
BENCH_RULE = (
    '한국어로 답하세요. 제공된 조건과 이번 관찰만 사용하세요. '
    '정보가 없으면 알 수 없다고 답하세요. '
    '출력은 답: ... / 근거: ... 형식으로 두 문장 이내로 쓰세요.'
)
BOX_RULE = (
    '판정 규칙: 빨간 박스 1개와 파란 박스 1개가 모두 보이고 색과 개수가 확인되면 '
    'clear(확인 완료)입니다. 전체 작업대가 보이며 박스가 없거나 개수가 다르면 '
    'caution(담당자 확인)입니다. 가림이나 정보 부족으로 색·개수를 확인할 수 없으면 '
    'unknown(확인 보류·담당자 확인)입니다. 가려짐은 없음과 다릅니다. '
)
bench_cases = [
    {'id': 'box_visible', 'history': [], 'question': BOX_RULE +
     '현재 관찰: 빨간 박스 1개와 파란 박스 1개가 모두 보이고 색과 개수가 확인됩니다. 판정은?',
     'expected': 'clear'},
    {'id': 'box_hidden', 'history': [], 'question': BOX_RULE +
     '현재 관찰: 빨간 박스 1개는 보이고 파란 박스는 상자에 가려져 있습니다. 판정은?',
     'expected': 'unknown'},
    {'id': 'box_missing', 'history': [], 'question': BOX_RULE +
     '현재 관찰: 전체 작업대가 보입니다. 빨간 박스 1개가 있고 파란 박스는 없습니다. 판정은?',
     'expected': 'caution'},
    {'id': 'box_no_observation', 'history': [], 'question': BOX_RULE +
     '현재 관찰 정보는 없습니다. 판정은?', 'expected': 'unknown'},
]
bench_prompt = ChatPromptTemplate.from_messages([
    ('system', BENCH_RULE), MessagesPlaceholder('history'), ('human', '{question}'),
])
print('박스 판단 문제 수:', len(bench_cases))
print('동일한 첫 입력:', bench_prompt.invoke({
    'history': bench_cases[0]['history'], 'question': bench_cases[0]['question'],
}).to_messages())

### 4-2. 먼저 Gemma 3로 판단하기

아래 셀은 **Gemma 3만 실행**합니다. 준비 요청 1회 후 박스 문제 4개를 순서대로 보냅니다. 응답에서 판정과 근거를 읽고 특히 가려짐을 없음으로 단정하지 않는지 확인하세요.

LangChain의 `bench_prompt | bench_model`로 요청합니다. temperature=0, seed=2026, num_ctx=4096, num_predict=256, reasoning=False를 사용합니다. `reasoning=False`는 뒤의 Gemma 4에도 동일하게 적용해 최종 답변을 비교하기 위한 설정입니다.

`seconds`는 전체 경과 시간이고 duration 메타데이터는 나노초입니다. 토큰/초는 `eval_count / (eval_duration / 1e9)`로 계산합니다. 정상 반환은 의미 정답 판정과 다릅니다.
![모델 실행 반복문과 성공·실패 기록 흐름](../../assets/02-flow-request.svg)

**실습 절차:** 이 셀을 한 번 실행하고 다섯 줄의 요청 결과가 끝날 때까지 기다립니다. warmup은 모델을 준비하는 요청이며 네 박스 판정에 포함하지 않습니다. 출력 마지막의 저장 경로를 확인한 뒤 각 박스의 답과 근거를 읽으세요.

**변수 따라가기:** requests는 준비 요청과 네 문제의 목록입니다. for 안의 case는 그중 한 문제, inputs는 모델에 보낼 값, row는 한 요청의 기록입니다. messages와 input_sha256은 입력 일치를 확인하는 값이며 정답 검사는 아닙니다.

**분기와 저장:** try는 호출·응답 검사를 수행합니다. except는 실패 원문을 기록하고, 그 뒤의 시간 계산·목록 추가·JSONL 저장은 성공과 실패 모두에서 실행됩니다. model_rows에는 다섯 기록이 쌓이고 마지막에 gemma3_rows로 보관합니다.

**다시 실행할 때:** 이 셀은 새로운 실행 폴더와 Gemma 3 결과를 만듭니다. 이 셀을 다시 실행했다면 Gemma 4와 비교 셀도 순서대로 다시 실행해 같은 실행의 결과를 비교하세요.


In [ ]:
import hashlib, statistics
bench_dir = ROOT / 'Data/outputs/04.LLM_v2' / ('boxes-' + str(time.time_ns()))
bench_dir.mkdir(parents=True, exist_ok=True)
bench_log = bench_dir / 'comparison.jsonl'
model_tag = 'gemma3:1b'
model_rows = []
bench_model = ChatOllama(
    model=model_tag, base_url=model.base_url,
    temperature=0, seed=2026, num_ctx=4096, num_predict=256,
    reasoning=False, keep_alive='5m', client_kwargs={'timeout': 180},
)
bench_chain = bench_prompt | bench_model
requests = [{'id': 'warmup', 'history': [], 'question': '준비되면 OK라고만 답하세요.',
             'expected': None}] + bench_cases
# [반복] 준비 요청 1개와 박스 문제 4개를 하나씩 처리합니다.
for case in requests:
    # [입력] question/history를 구성하고 원문·해시를 기록합니다.
    inputs = {'history': case['history'], 'question': case['question']}
    messages = [m.model_dump(mode='json') for m in bench_prompt.invoke(inputs).to_messages()]
    digest = hashlib.sha256(json.dumps(messages, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    row = {'model': model_tag, 'repeat': 1, 'case': case['id'],
           'expected': case['expected'], 'messages': messages, 'input_sha256': digest,
           'settings': {'temperature': 0, 'seed': 2026, 'num_ctx': 4096, 'num_predict': 256, 'reasoning': False},
           'correct': None}  # 응답을 보고 사람이 평가합니다.
    started = time.perf_counter()
    # [호출·검사] 이 구간에서만 실제 모델 요청을 보냅니다.
    try:
        reply = bench_chain.invoke(inputs)
        md = reply.response_metadata
        row.update(answer=reply.content, metadata=md, usage=reply.usage_metadata)
        row['load_seconds'] = md.get('load_duration', 0) / 1e9
        row['generation_seconds'] = md.get('eval_duration', 0) / 1e9
        row['output_tokens'] = md.get('eval_count')
        row['tokens_per_second'] = (row['output_tokens'] / row['generation_seconds']
            if row['output_tokens'] is not None and row['generation_seconds'] > 0 else None)
        if not reply.content or md.get('done_reason') == 'length':
            raise ValueError('빈 응답 또는 출력 상한으로 잘린 응답')
        row['status'] = 'ok'
    except Exception as exc:
        row.update(status='request_error', error=str(exc))
    # [공통 저장] 성공·실패에 관계없이 시간을 재고 로그를 남깁니다.
    row['seconds'] = time.perf_counter() - started
    model_rows.append(row)
    with bench_log.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(model_tag, case['id'], row['status'],
          round(row['seconds'], 2), row.get('answer'), flush=True)
gemma3_rows = model_rows
print('Gemma 3 결과 저장:', bench_log)

### 4-3. Gemma 4를 추가해 같은 문제 판단하기

Gemma 3의 네 응답을 확인한 뒤 아래 셀로 넘어갑니다. **모델 태그만 `gemma4:e2b-it-qat`로 바꾸고** 같은 메시지·판정 규칙·생성 설정을 사용합니다. 준비 요청 1회와 박스 문제 4회를 실행합니다.

먼저 Jetson **호스트 터미널**에서 모델 목록을 확인합니다. 없는 경우에만 담당자 한 명이 다운로드합니다. 모델이 준비되면 개인 Jupyter에서 아래 Python 셀을 실행합니다.

```bash
docker exec physical_ai_ollama ollama list
# 모델이 없을 때만
docker exec physical_ai_ollama ollama pull gemma4:e2b-it-qat
```

Gemma 4의 thinking 기본 동작에서는 짧은 생성 상한 안에 최종 답변이 나오지 않을 수 있어 `reasoning=False`로 비교합니다. 모델별 토크나이저와 직렬화는 다를 수 있으며 동일 입력은 같은 역할·내용의 메시지를 뜻합니다.
**실습 절차:** Gemma 3 원문을 먼저 확인하고 이 셀을 실행합니다. 앞 셀의 gemma3_rows가 준비되지 않았다면 첫 검사에서 멈춥니다. 이때 비교 코드만 실행하지 말고 Gemma 3 셀부터 완료하세요.

**같은 것과 바뀌는 것:** bench_cases·bench_prompt·생성 설정은 같고 model_tag만 Gemma 4로 바뀝니다. 내부 for/try 흐름은 앞 그림과 같습니다. 새 model_rows를 gemma4_rows로 보관하고 두 결과 목록을 bench_rows로 합칩니다. Gemma 3의 기록을 덮어쓰지 않습니다.

**완료 확인:** 준비 포함 다섯 결과와 저장 경로가 출력됐는지 봅니다. 이 셀만 다시 실행하면 현재 Gemma 4 목록은 교체되지만 JSONL에는 재실행 기록이 추가됩니다. 비교 셀은 현재 메모리의 두 목록을 사용합니다.


In [ ]:
# 앞 Gemma 3 실행 셀의 입력·설정을 그대로 사용합니다.
if len(gemma3_rows) != len(bench_cases) + 1:
    raise RuntimeError('Gemma 3 실행 셀을 먼저 완료하세요.')
model_tag = 'gemma4:e2b-it-qat'
model_rows = []
bench_model = ChatOllama(
    model=model_tag, base_url=model.base_url,
    temperature=0, seed=2026, num_ctx=4096, num_predict=256,
    reasoning=False, keep_alive='5m', client_kwargs={'timeout': 180},
)
bench_chain = bench_prompt | bench_model
requests = [{'id': 'warmup', 'history': [], 'question': '준비되면 OK라고만 답하세요.',
             'expected': None}] + bench_cases
# [반복] 준비 요청 1개와 박스 문제 4개를 하나씩 처리합니다.
for case in requests:
    # [입력] question/history를 구성하고 원문·해시를 기록합니다.
    inputs = {'history': case['history'], 'question': case['question']}
    messages = [m.model_dump(mode='json') for m in bench_prompt.invoke(inputs).to_messages()]
    digest = hashlib.sha256(json.dumps(messages, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    row = {'model': model_tag, 'repeat': 1, 'case': case['id'],
           'expected': case['expected'], 'messages': messages, 'input_sha256': digest,
           'settings': {'temperature': 0, 'seed': 2026, 'num_ctx': 4096, 'num_predict': 256, 'reasoning': False},
           'correct': None}  # 응답을 보고 사람이 평가합니다.
    started = time.perf_counter()
    # [호출·검사] 이 구간에서만 실제 모델 요청을 보냅니다.
    try:
        reply = bench_chain.invoke(inputs)
        md = reply.response_metadata
        row.update(answer=reply.content, metadata=md, usage=reply.usage_metadata)
        row['load_seconds'] = md.get('load_duration', 0) / 1e9
        row['generation_seconds'] = md.get('eval_duration', 0) / 1e9
        row['output_tokens'] = md.get('eval_count')
        row['tokens_per_second'] = (row['output_tokens'] / row['generation_seconds']
            if row['output_tokens'] is not None and row['generation_seconds'] > 0 else None)
        if not reply.content or md.get('done_reason') == 'length':
            raise ValueError('빈 응답 또는 출력 상한으로 잘린 응답')
        row['status'] = 'ok'
    except Exception as exc:
        row.update(status='request_error', error=str(exc))
    # [공통 저장] 성공·실패에 관계없이 시간을 재고 로그를 남깁니다.
    row['seconds'] = time.perf_counter() - started
    model_rows.append(row)
    with bench_log.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(model_tag, case['id'], row['status'],
          round(row['seconds'], 2), row.get('answer'), flush=True)
gemma4_rows = model_rows
bench_rows = gemma3_rows + gemma4_rows
print('두 모델 결과 저장:', bench_log)

### 4-4. 두 모델의 박스 판단 비교

준비 요청을 제외한 모델별 네 응답을 나란히 읽습니다. 정상은 clear, 가림은 unknown, 전체 시야에서 누락은 caution, 관찰 정보 없음은 unknown이 기대 판정입니다. 판정과 근거가 모순되면 정답으로 인정하지 않습니다.

`correct=None`은 미평가입니다. 각 응답을 읽고 True/False를 기록한 뒤 요약 셀을 다시 실행하세요. 전체 10회 요청 중 채점 대상은 8회입니다. Gemma 3 다음 Gemma 4로 진행하는 수업 실습이며, 모델 순서·캐시·공용 서버 부하를 통제한 성능 벤치마크는 아닙니다. 시간·출력 길이·정확도를 함께 비교합니다.

[강사 실행 비교 결과](../../MODEL_COMPARISON.md)
![비교 코드의 입력 검사와 평가·저장 흐름](../../assets/02-flow-review.svg)

**실습 절차:** 먼저 비교 셀을 실행해 문제별 두 모델 원문을 읽습니다. measured는 준비 요청을 뺀 여덟 기록입니다. 같은 문제의 input_sha256이 다르면 assert에서 멈추므로 입력을 바꾼 뒤 한 모델만 다시 실행하지 않았는지 확인하세요.

**사람 평가 기록 예시:** 아래 비교 셀을 한 번 실행한 다음 새 코드 셀에서 평가한 행만 수정할 수 있습니다.

```python
for row in measured:
    if row['model'] == 'gemma3:1b' and row['case'] == 'box_hidden':
        row['correct'] = False  # 실제 원문을 보고 정합니다.
        row['review_note'] = '판정과 관찰 근거를 직접 기록'
```

평가 후 비교 셀을 다시 실행하면 같은 row 사전의 평가를 유지한 채 요약과 review.json을 갱신합니다. 모델 재호출은 없습니다. graded=0은 아직 평가한 행이 없다는 뜻이며 오답 0개라는 뜻이 아닙니다. 실패 응답도 False로 평가해 제외하지 마세요.


In [ ]:
measured = [r for r in bench_rows if r['case'] != 'warmup']
for case in bench_cases:
    case_rows = [r for r in measured if r['case'] == case['id']]
    assert len({r['input_sha256'] for r in case_rows}) == 1, '모델에 전달한 입력이 다릅니다.'
    print('\n', case['id'], '기대:', case['expected'])
    for row in case_rows:
        print(row['model'], row['repeat'], row['status'], row['answer'] if 'answer' in row else row['error'])

for model_tag in ['gemma3:1b', 'gemma4:e2b-it-qat']:
    rows = [r for r in measured if r['model'] == model_tag]
    rates = [r['tokens_per_second'] for r in rows if r.get('tokens_per_second') is not None]
    graded = [r for r in rows if isinstance(r['correct'], bool)]
    print({'model': model_tag, 'requests': len(rows),
           'request_errors': sum(r['status'] != 'ok' for r in rows),
           'median_seconds': statistics.median(r['seconds'] for r in rows),
           'median_tokens_per_second': statistics.median(rates) if rates else None,
           'graded': len(graded), 'correct': sum(r['correct'] for r in graded)})
(bench_dir / 'review.json').write_text(json.dumps(measured, ensure_ascii=False, indent=2))

## 5. 기억 × 지시의 네 구성을 같은 질문으로 비교

네 구성 모두 같은 두 턴을 사용합니다. 각 구성은 빈 이력으로 시작합니다. memory=False에서는 두 번째 요청에 첫 턴의 조건이 실제로 없는지 확인하세요. 아래는 실제 모델 요청 8회입니다. 추가 비교로 제공하며 기본 실습이 끝난 뒤 필요한 경우 RUN_FOUR_WAY=True로 실행합니다.
![기억과 단계 지시의 중첩 반복 구조](../../assets/02-flow-fourway.svg)

**코드 읽기:** 바깥 for는 memory, 다음 for는 steps를 바꿉니다. 조합마다 compare_prompt·compare_chain을 만들고 compare_history=[]로 시작합니다. 가장 안쪽 for가 두 질문을 보냅니다. memory=False이면 두 번째 질문에도 이력을 보내지 않습니다. steps=True이면 STEP_RULE만 추가합니다.

**실행 확인:** 실행 전 RUN_FOUR_WAY의 현재 값을 직접 확인하세요. True면 실제 요청 여덟 회, False면 안내 출력 후 건너뜁니다. 이 실험은 위의 Gemma 4 결과를 이어 쓰지 않고 준비 셀의 model을 사용합니다. 조합별로 실제 전송된 messages와 응답을 함께 읽으세요.


In [ ]:
RUN_FOUR_WAY = False
four_way = []
if RUN_FOUR_WAY:
    for memory in [False, True]:
        for steps in [False, True]:
            compare_prompt = ChatPromptTemplate.from_messages([
                ('system', BASE_RULE + (' ' + STEP_RULE if steps else '')),
                MessagesPlaceholder('history'), ('human', '{question}'),
            ])
            compare_chain = compare_prompt | model
            compare_history = []
            for question in [
                '확인 완료 조건은 빨간 블록 1개와 파란 블록 1개가 모두 보이는 것입니다.',
                '지금 파란 블록은 상자에 가려져 있습니다. 확인 완료로 기록할까요?',
            ]:
                inputs = {'history': compare_history if memory else [], 'question': question}
                started = time.perf_counter()
                row = {'memory': memory, 'steps': steps, 'question': question,
                       'messages': [m.model_dump(mode='json') for m in compare_prompt.invoke(inputs).to_messages()]}
                try:
                    response = compare_chain.invoke(inputs)
                    row.update(answer=response.content, metadata=response.response_metadata)
                    if not response.content or response.response_metadata.get('done_reason') == 'length':
                        raise ValueError('빈 응답 또는 잘린 응답')
                    row['status'] = 'ok'
                    if memory:
                        compare_history = (compare_history + [HumanMessage(question), AIMessage(response.content)])[-6:]
                except Exception as exc:
                    row.update(status='request_error', error=str(exc))
                row['seconds'] = time.perf_counter() - started
                four_way.append(row)
                print(memory, steps, row['status'], row.get('answer'), row.get('error'))
else:
    print('추가 2×2 비교는 실행하지 않았습니다.')

## 6. 초기화와 오류를 서버 없이 확인

history.clear()는 Python 목록을 비웁니다. 모델 파일이나 별도 저장 파일을 삭제하지 않습니다. 모의 실패를 사용하면 실제 서버 장애를 만들지 않고 이력 갱신 위치를 확인할 수 있습니다.
**실습 절차:** before에 history의 사본을 보관합니다. fail_model은 실제 서버 대신 TimeoutError를 내므로 except로 이동해야 합니다. 이후 assert가 이력이 바뀌지 않았는지 확인합니다. 마지막 clear 이후 빈 목록이 되는지 확인하세요. 이 셀을 실행해도 저장한 박스 로그나 모델 파일은 삭제되지 않습니다.


In [ ]:
def fail_model(value):
    raise TimeoutError('모의 timeout')
failure_chain = history_prompt | RunnableLambda(fail_model)
before = list(history)
try:
    response = failure_chain.invoke({'history': history, 'question': '실패 예제'})
except TimeoutError:
    print('모의 실패: 기존 이력 보존')
else:
    history += [HumanMessage('실패 예제'), AIMessage(response.content)]
assert history == before
history.clear()
assert history == []
print('대화 목록 초기화 완료')

## 7. 반환값과 평가

prompt.invoke는 ChatPromptValue, chain.invoke는 AIMessage입니다. 기억 유무는 전송 messages에서 확인합니다. 답변의 조건 준수·근거·지연은 실제 결과를 읽고 기록합니다. 실행하지 않은 추가 비교를 완료했다고 표시하지 않습니다.

## 마지막 TODO — 최근 대화 이력 선택

**코드 작성: 있음 · 10분 · 수업 시간 안에 진행**

수정 파일: [practice/chat_history.py](../../practice/chat_history.py). JupyterLab 편집기에서 함수를 완성하고 저장한 뒤 아래 로딩 셀과 검사 셀을 다시 실행하세요. 노트북은 학생 파일을 덮어쓰지 않습니다.

**입력·출력 계약:** history는 human/ai 메시지가 번갈아 들어 있는 완전한 대화 쌍 목록입니다. enabled=False면 빈 목록, True면 최근 max_pairs개 쌍의 깊은 복사본을 반환하세요. max_pairs는 1 이상의 정수이며 bool·0·음수는 ValueError입니다. 원본 목록과 메시지를 바꾸지 않습니다.

**힌트:** 대화 한 쌍은 메시지 두 개입니다. [-2 * max_pairs:]와 copy.deepcopy를 조합하세요.

미완성이면 마지막 검사에서 NotImplementedError가 나는 것이 정상입니다. 검사 통과 후 아래 적용 예제의 중간값을 설명하세요. 이 예제는 학생 함수를 직접 사용하며 본문의 API 실행 셀을 자동 교체하지 않습니다.
![TODO 수정부터 로딩·검사·적용까지](../../assets/02-flow-todo.svg)

**진행 순서:** 노트북 코드가 아니라 링크된 practice/chat_history.py를 편집해 저장합니다. 첫 로딩 셀을 다시 실행해 선택 구현과 함수 소스를 확인한 다음 검사 셀을 실행합니다. 파일만 저장하고 로딩을 건너뛰면 메모리의 task가 이전 함수를 가리킬 수 있습니다.

**검사 후:** 적용 예제의 원본 history와 selected를 비교하세요. 최근 한 쌍을 요청했을 때 human/ai 두 메시지가 선택되는지, 원본이 변하지 않았는지 확인합니다. 출력된 프롬프트는 입력 확인용이며 모델을 호출하지 않습니다.


In [ ]:
import importlib.util, inspect, os
runner_path=COURSE/'practice/runner.py'
spec=importlib.util.spec_from_file_location('day2_todo_runner',runner_path)
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
VARIANT=os.environ.get('DAY2_TODO_VARIANT','exercise')
task=runner.load('chat_history',VARIANT)
print('선택 구현:',VARIANT)
print(inspect.getsource(task.select_history))

In [ ]:
print(runner.check('chat_history',task))

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
history=[HumanMessage('작업 코드 BLUE-27'),AIMessage('확인'),HumanMessage('파란 블록 가림'),AIMessage('확인 보류')]
selected=task.select_history(history, True, 1)
prompt=ChatPromptTemplate.from_messages([('system','짧게 답하세요.'),MessagesPlaceholder('history'),('human','{question}')])
messages=prompt.invoke({'history':selected,'question':'그럼 무엇을 확인하나요?'})
print([(m.type,m.content) for m in messages.to_messages()])

### TODO 완료와 참고 정답

- [ ] 함수를 직접 작성하고 검사 셀을 통과했습니다.
- [ ] 경계 입력 한 개를 추가해 예상 결과를 설명했습니다.
- [ ] 적용 예제의 입력과 반환값을 설명했습니다.

강사가 정한 직접 시도 시간 뒤 [참고 정답](../../practice/solutions/chat_history.py)을 비교하세요. 참고 정답을 실행했다고 학생 파일이 완성되는 것은 아닙니다.

**해설:** 메시지 두 개를 한 쌍으로 유지해야 질문과 답의 순서가 보존됩니다. 깊은 복사를 하면 반환된 메시지의 내용을 바꾸어도 원래 history가 바뀌지 않습니다.

추가 모델 요청 없이 함수와 데이터 흐름을 검사합니다. 실제 모델·카메라 결과 평가는 본문 실습에서 수행합니다.

## 완료 확인

실제 입력과 출력 기록을 확인하고, 형식·근거·지연을 구분해 설명합니다. 오류는 숨기지 않고 원문과 함께 남깁니다. 추가 실험은 한 조건만 바꿔 비교합니다.